In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from shapely.geometry import box
from matplotlib.patches import Patch
from pathlib import Path
from shapely.ops import unary_union

In [ ]:
# setup
path = '~/Desktop/Desktop/epidemiology_PhD/00_repos/la-wf/01_data/01_raw/'

# read in geojson
evac_lp = gpd.read_file(path + 'jan_8_boundaries.geojson')
evac_hmb = gpd.read_file(path + 'california_active_evacuation_zones_20250113_193346.geojson')
evac_kenneth = gpd.read_file(path + 'kenneth_ventura_evac/kenneth_ventura_evac.shp')

# palette
okeefe = ["#fbe3c2", "#f2c88f", "#ecb27d", "#e69c6b", "#d37750", "#b9563f", "#611F10"]

# read in ca cts and subset to the so cal catchment area 
so_cal_counties = ["025", "029", "037", "065", "059", "071", "073", "083", "111", "079"]
cts_ca = gpd.read_file(path + "tl_2010_06_tract10.shp")
cts_kp = cts_ca[cts_ca['COUNTYFP10'].isin(so_cal_counties)]

# read in zctas 
zctas_us = gpd.read_file(path + "tl_2020_us_zcta520.shp")

# intersect zctas with the subsetted cts to get the zcta catchment area
zctas_kp = gpd.sjoin(zctas_us, cts_kp, how="inner", predicate="intersects")
zctas_kp = zctas_kp[["ZCTA5CE20", "geometry"]].rename(columns={"ZCTA5CE20": "zcta"})
zctas_kp = zctas_kp.dissolve(by='zcta', as_index=True)

# pm 
pm_df = gpd.read_file(path + 'childs_pm/pm_poly.geojson')


In [ ]:
evac_kenneth_mercator = evac_kenneth.to_crs(epsg=3857)

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8))

# Plot the polygon first
evac_kenneth_mercator.plot(ax=ax, 
                          facecolor='red', 
                          edgecolor='darkred', 
                          alpha=0.6, 
                          linewidth=2)

# Add the basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

# Set the extent based on the reprojected data
ax.set_xlim(evac_kenneth_mercator.total_bounds[0], evac_kenneth_mercator.total_bounds[2])
ax.set_ylim(evac_kenneth_mercator.total_bounds[1], evac_kenneth_mercator.total_bounds[3])

# Styling
ax.set_title("Kenneth Evacuation Zones", fontsize=20, pad=20)
ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# helper functions

def parse_datetime_string(date_str):
    # split date and time
    date_part = str(date_str)[:7]    # Gets '2025012'
    
    # parse date
    year = date_part[:4]       # '2025'
    day = date_part[5:]        # '012' -> '12'
    month = '01'               # January
    
    # construct datetime string 
    datetime_str = f"{year}-{month}-{day}"
    return pd.to_datetime(datetime_str)

def remove_overlaps(smoke_df):
    smoke_df = smoke_df.to_crs(epsg=3857)
    
    # split by density and make copies of the data
    heavy = smoke_df[smoke_df['Density'] == 'Heavy'].copy()
    medium = smoke_df[smoke_df['Density'] == 'Medium'].copy()
    light = smoke_df[smoke_df['Density'] == 'Light'].copy()
    
    # lets split them up! 
    if not medium.empty and not heavy.empty:
        medium['geometry'] = gpd.GeoSeries(
            # remove heavy areas from medium
            medium.geometry.apply(lambda x: x.difference(heavy.geometry.union_all())),
            index=medium.index,
            crs=medium.crs
        )
    
    if not light.empty:
        if not heavy.empty:
            # remove heavy areas from light
            light['geometry'] = gpd.GeoSeries(
                light.geometry.apply(lambda x: x.difference(heavy.geometry.union_all())),
                index=light.index,
                crs=light.crs
            )
        if not medium.empty:
            # remove medium areas
            light['geometry'] = gpd.GeoSeries(
                light.geometry.apply(lambda x: x.difference(medium.geometry.union_all())),
                index=light.index,
                crs=light.crs
            )
    
    # come back together
    return pd.concat([light, medium, heavy])

In [ ]:
# clip to logan's bounds because they are specific to the la fires
# use logan's bounds since his is specific to LA fires
bounds_lp = evac_lp.total_bounds
minx, miny, maxx, maxy = bounds_lp

bbox = box(minx, miny, maxx, maxy)
bbox_gdf = gpd.GeoDataFrame({'geometry': [bbox]}, crs=evac_lp.crs)

# Clip evac_hmb to the bounds of evac_lp
evac_hmb_clipped = gpd.clip(evac_hmb, bbox_gdf)

# for contextily basemap, we need to use web mercator projection (EPSG:3857)
if evac_lp.crs != 'EPSG:3857':
    evac_lp_webmerc = evac_lp.to_crs(epsg=3857)
    evac_hmb_webmerc = evac_hmb_clipped.to_crs(epsg=3857)
else:
    evac_lp_webmerc = evac_lp
    evac_hmb_webmerc = evac_hmb_clipped

# make plot
fig, ax = plt.subplots(figsize=(10, 7))

evac_lp_webmerc.plot(ax=ax, color=okeefe[1], alpha=0.7, edgecolor='black', linewidth=0.5, label='LP boundaries')
evac_hmb_webmerc.plot(ax=ax, color=okeefe[5], alpha=0.7, edgecolor='black', linewidth=0.5, label='HMB boundaries')
evac_kenneth_mercator.plot(ax=ax, 
                          facecolor=okeefe[6], 
                          edgecolor='black', 
                          alpha=0.6, 
                          linewidth=2)

legend_elements = [
    Patch(facecolor=okeefe[1], edgecolor='black', alpha=0.7, label='LP boundaries'),
    Patch(facecolor=okeefe[5], edgecolor='black', alpha=0.7, label='HMB boundaries'),
    Patch(facecolor=okeefe[6], edgecolor='black', alpha=0.6, label='Kenneth evac')
]


ax.legend(handles=legend_elements, loc='upper left', frameon=True)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
plt.title('evac boundaries: overlay')

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# Create a figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7), sharey=True)

# Get bounds in web mercator for proper display with contextily
bounds_webmerc = evac_lp_webmerc.total_bounds
minx_web, miny_web, maxx_web, maxy_web = bounds_webmerc

# plot 1: LP
evac_lp_webmerc.plot(ax=ax1, color=okeefe[1], alpha=0.7, edgecolor='black', linewidth=0.5)
ctx.add_basemap(ax1, source=ctx.providers.CartoDB.Positron)
ax1.set_title('LP evac zones', fontsize=14)
ax1.set_xlim(minx_web, maxx_web)
ax1.set_ylim(miny_web, maxy_web)
ax1.set_axis_off()

# plot 2: HMB
evac_hmb_webmerc.plot(ax=ax2, color=okeefe[5], alpha=0.7, edgecolor='black', linewidth=0.5)
ctx.add_basemap(ax2, source=ctx.providers.CartoDB.Positron)
ax2.set_title('HMB evac zones', fontsize=14)
ax2.set_xlim(minx_web, maxx_web)  # Fixed: ax2 instead of ax1
ax2.set_ylim(miny_web, maxy_web)  # Fixed: ax2 instead of ax1
ax2.set_axis_off()

plt.suptitle('evac boundaries: side by side', fontsize=20, y=0.95)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7), sharey=True)

bounds_webmerc = evac_lp_webmerc.total_bounds
minx_web, miny_web, maxx_web, maxy_web = bounds_webmerc

# PLOT 1: LP
lp_colors = {
    'Mandatory': okeefe[1],  # orig color (for mandatory)
    'Warning': okeefe[2]      # one index down in palette for warning
}

# empty lists to collect patches for the legend
lp_handles = []

# plot each type with its own color
for evac_type, data in evac_lp_webmerc.groupby('type'):
    data.plot(
        ax=ax1, 
        color=lp_colors[evac_type], 
        alpha=0.7, 
        edgecolor='black', 
        linewidth=0.5
    )
    # create legend
    lp_handles.append(Patch(facecolor=lp_colors[evac_type], edgecolor='black', 
                          alpha=0.7, label=evac_type))

ctx.add_basemap(ax1, source=ctx.providers.CartoDB.Positron)
ax1.set_title('LP evac zones', fontsize=14)
ax1.set_xlim(minx_web, maxx_web)
ax1.set_ylim(miny_web, maxy_web)
ax1.legend(handles=lp_handles, title='Evacuation Type', loc='upper left', frameon=True, framealpha=0.9)
ax1.set_axis_off()

# plot 2: HMB 

hmb_colors = {
    'Evacuation Order': okeefe[5],    # orig color for evacuation order
    'Evacuation Warning': okeefe[6]   # one index down for evacuation warning
}

# empty lists to collect patches for the legend
hmb_handles = []

# plot each status with its own color
for status, data in evac_hmb_webmerc.groupby('STATUS'):
    data.plot(
        ax=ax2, 
        color=hmb_colors[status], 
        alpha=0.7, 
        edgecolor='black', 
        linewidth=0.5
    )
    # create legend
    hmb_handles.append(Patch(facecolor=hmb_colors[status], edgecolor='black', 
                           alpha=0.7, label=status))

ctx.add_basemap(ax2, source=ctx.providers.CartoDB.Positron)
ax2.set_title('HMB evac zones', fontsize=14)
ax2.set_xlim(minx_web, maxx_web)
ax2.set_ylim(miny_web, maxy_web)
ax2.legend(handles=hmb_handles, title='Evacuation Status', loc='upper left', frameon=True, framealpha=0.9)
ax2.set_axis_off()

plt.suptitle('evac boundaries: side by side', fontsize=20, y=0.95)

plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()

In [ ]:
# plot pm data alone 

pm_data_webmerc = pm_df.to_crs(epsg=3857)

# union of all PM polygons
pm_union = gpd.GeoDataFrame(
    geometry=[unary_union(pm_data_webmerc.geometry)],
    crs=pm_data_webmerc.crs
)

fig, ax = plt.subplots(figsize=(8, 8))

pm_union.plot(
    ax=ax,
    color='gray',
    alpha=0.6,
    edgecolor='black',
    linewidth=0.5
)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
bounds = pm_union.total_bounds
buffer_x = (bounds[2] - bounds[0]) * 0.05
buffer_y = (bounds[3] - bounds[1]) * 0.05
minx, miny, maxx, maxy = bounds[0] - buffer_x, bounds[1] - buffer_y, bounds[2] + buffer_x, bounds[3] + buffer_y
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

ax.set_title('pm', fontsize=16)
ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# plot pm plus evac zones

# bounds of the pm union with a small buffer
pm_bounds = pm_union.total_bounds
buffer_x = (pm_bounds[2] - pm_bounds[0]) * 0.05
buffer_y = (pm_bounds[3] - pm_bounds[1]) * 0.05
minx, miny, maxx, maxy = pm_bounds[0] - buffer_x, pm_bounds[1] - buffer_y, pm_bounds[2] + buffer_x, pm_bounds[3] + buffer_y

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), sharey=True)

# PLOT 1: LP evacuation zones with PM plume
# -----------------------------------------

# plot pm 
pm_union.plot(
    ax=ax1,
    color='gray',
    alpha=0.6,
    edgecolor='black',
    linewidth=0.5,
    zorder=5  # middle layer
)

# add basemap (setting a low z-order ensures it's at the bottom)
ctx.add_basemap(ax1, source=ctx.providers.CartoDB.Positron, zorder=1)

lp_colors = {
    'Mandatory': okeefe[1],  # orig color (for mandatory)
    'Warning': okeefe[2]      # one index down in palette for warning
}

# empty list to collect patches for the legend
lp_handles = []

# plot each evac type with its own color (with higher z-order to be on top)
for evac_type, data in evac_lp_webmerc.groupby('type'):
    data.plot(
        ax=ax1, 
        color=lp_colors[evac_type], 
        alpha=0.7, 
        edgecolor='black', 
        linewidth=0.5,
        zorder=10  # Top layer
    )
    # legend
    lp_handles.append(Patch(facecolor=lp_colors[evac_type], edgecolor='black', 
                          alpha=0.7, label=evac_type))

# add pm to legend
lp_handles.append(Patch(facecolor='gray', edgecolor='black', 
                      alpha=0.6, label='pm'))

ax1.set_title('LP evac zones', fontsize=14)

ax1.set_xlim(minx, maxx)
ax1.set_ylim(miny, maxy)
ax1.legend(handles=lp_handles, title='legend', loc='upper left', 
           frameon=True, framealpha=0.9)
ax1.set_axis_off()

# PLOT 2: hmb evacuation zones with pm 
# ------------------------------------------

# plot the pm with controlled z-order
pm_union.plot(
    ax=ax2,
    color='gray',
    alpha=0.6,
    edgecolor='black',
    linewidth=0.5,
    zorder=5  # Middle layer
)

# add basemap (setting a low z-order ensures it's at the bottom)
ctx.add_basemap(ax2, source=ctx.providers.CartoDB.Positron, zorder=1)

hmb_colors = {
    'Evacuation Order': okeefe[5],    # orig color for evacuation order
    'Evacuation Warning': okeefe[6]   # one index down for evacuation warning
}

# empty list to collect patches for the legend
hmb_handles = []

# plot each status with its own color (with higher z-order to be on top)
for status, data in evac_hmb_webmerc.groupby('STATUS'):
    data.plot(
        ax=ax2, 
        color=hmb_colors[status], 
        alpha=0.7, 
        edgecolor='black', 
        linewidth=0.5,
        zorder=10 # top layer
    )
    # legend
    hmb_handles.append(Patch(facecolor=hmb_colors[status], edgecolor='black', 
                           alpha=0.7, label=status))

# add pm to legend
hmb_handles.append(Patch(facecolor='gray', edgecolor='black', 
                       alpha=0.6, label='pm'))

ax2.set_title('HMB evac zones', fontsize=14)

ax2.set_xlim(minx, maxx)
ax2.set_ylim(miny, maxy)
ax2.legend(handles=hmb_handles, title='legend', loc='upper left', 
           frameon=True, framealpha=0.9)
ax2.set_axis_off()

plt.suptitle('evac zones with pm', fontsize=20, y=0.95)
plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()

## note: subset out the polygons with no pm (where interp > XX [ASK JOAN] and use all dates but average over the first week (jan 7-13))
# if there is no value in the dataset it is bc they are not under a smoke plume nor plausibly affected by the fires
# dont store things not under smoke plumes, but do store 0s if they are under plumes but have pm = 0
# if its in the dataset, it was under a plume and under a high split trajectory point of the eaton fire and include all interpolated values for those
# instead maybe we use the column c station pm and draw a cone 

In [ ]:
# plot using smoke plume data

# read in smoke files for jan 7-12, 2025
smoke_dir = Path(path).expanduser() / "smoke_plume"
smoke_files = list(smoke_dir.glob("*/*.shp"))

# read in all files and create a date var for each: 
smoke = pd.concat([gpd.read_file(f) for f in smoke_files])
smoke["date"] = smoke["Start"].apply(parse_datetime_string)

# subset to only the density, geometry, and date
smoke = smoke[["date", "geometry", "Density"]]

# subset smoke to kp zctas
smoke_kp = smoke.to_crs(epsg=3857)
zctas_kp = zctas_kp.to_crs(epsg=3857)
smoke_kp = gpd.sjoin(smoke_kp, zctas_kp, how="inner", predicate="intersects")

# dissolve by date and density
smoke = smoke.dissolve(by=['date', 'Density'], as_index=True).reset_index()

# convert back to gdf
smoke = gpd.GeoDataFrame(smoke, geometry='geometry')

In [ ]:
# plot smoke plumes

okeefe = ["#fbe3c2", "#f2c88f", "#ecb27d", "#e69c6b", "#d37750", "#b9563f", "#92351e"]
density_colors = {
    'Light': okeefe[0],    # lightest
    'Medium': okeefe[2],   # medium
    'Heavy': okeefe[4]     # darkest
}

fig, ax = plt.subplots(figsize=(9, 7))

# subset to jan 8
all_dates = sorted(smoke["date"].unique())
jan8_date = None
for date in all_dates:
    if pd.to_datetime(date).strftime('%Y-%m-%d')[-5:] == '01-08':
        jan8_date = date
        break

if jan8_date is None:
    print("January 8th data not found.")
    if len(all_dates) > 1:
        jan8_date = all_dates[1]
    else:
        jan8_date = all_dates[0]  

# subset to this date
smoke_date = smoke[smoke["date"] == jan8_date]

# remove overlaps so that we can get heavy on top!
smoke_date_no_overlap = remove_overlaps(smoke_date)

    
# base map layers

# plot each density level separately
for density in ['Light', 'Medium', 'Heavy']:
    density_data = smoke_date_no_overlap[smoke_date_no_overlap['Density'] == density]
    if not density_data.empty:
        density_data.plot(
            ax=ax,
            color=density_colors[density],
            alpha=0.75,
            edgecolor=None
        )

# manual legend
legend_elements = [
    Patch(facecolor=density_colors['Heavy'], label='Heavy', alpha=0.75),
    Patch(facecolor=density_colors['Medium'], label='Medium', alpha=0.75),
    Patch(facecolor=density_colors['Light'], label='Light', alpha=0.75)
]
ax.legend(handles=legend_elements, loc='upper right')

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Voyager)
ax.set_axis_off()
ax.set_title(pd.to_datetime(date).strftime('%Y-%m-%d'))

# set the same bounds for each subplot
total_bounds = zctas_kp.total_bounds
ax.set_xlim(total_bounds[0], total_bounds[2])
ax.set_ylim(total_bounds[1], total_bounds[3])

plt.tight_layout()
plt.show()

In [ ]:
# plot smoke plumes plus evac zones 

# colors
okeefe = ["#fbe3c2", "#f2c88f", "#ecb27d", "#e69c6b", "#d37750", "#b9563f", "#92351e"]
density_colors = {
    'Light': okeefe[0],    # lightest
    'Medium': okeefe[2],   # medium
    'Heavy': okeefe[4]     # darkest
}

fig, ax = plt.subplots(figsize=(9, 7))

# subset to jan 8
all_dates = sorted(smoke["date"].unique())
jan8_date = None
for date in all_dates:
    if pd.to_datetime(date).strftime('%Y-%m-%d')[-5:] == '01-08':
        jan8_date = date
        break

if jan8_date is None:
    print("January 8th data not found.")
    if len(all_dates) > 1:
        jan8_date = all_dates[1]
    else:
        jan8_date = all_dates[0]  

# subset to this date
smoke_date = smoke[smoke["date"] == jan8_date]

# remove overlaps so that we can get heavy on top!
smoke_date_no_overlap = remove_overlaps(smoke_date)

# plot each density level separately
for density in ['Light', 'Medium', 'Heavy']:
    density_data = smoke_date_no_overlap[smoke_date_no_overlap['Density'] == density]
    if not density_data.empty:
        density_data.plot(
            ax=ax,
            color=density_colors[density],
            alpha=0.75,
            edgecolor=None
        )

# union of all evacuation zones
if 'evac_lp_webmerc' in locals() or 'evac_lp_webmerc' in globals():
    all_geometries = evac_lp_webmerc['geometry'].tolist()
    
    evac_union = unary_union(all_geometries)
    
    evac_union_gdf = GeoDataFrame(geometry=[evac_union])
    
    evac_union_gdf.plot(
        ax=ax,
        facecolor='none',      # No fill
        edgecolor='black',     # Black outline
        linewidth=1.5,         # Slightly thicker for visibility
        zorder=20              # On top of smoke layers
    )
    
    legend_elements = [
        Patch(facecolor=density_colors['Heavy'], label='Heavy Smoke', alpha=0.75),
        Patch(facecolor=density_colors['Medium'], label='Medium Smoke', alpha=0.75),
        Patch(facecolor=density_colors['Light'], label='Light Smoke', alpha=0.75),
        Patch(facecolor='none', edgecolor='black', linewidth=1.5, label='Evacuation Zone')
    ]
else:
    legend_elements = [
        Patch(facecolor=density_colors['Heavy'], label='Heavy Smoke', alpha=0.75),
        Patch(facecolor=density_colors['Medium'], label='Medium Smoke', alpha=0.75),
        Patch(facecolor=density_colors['Light'], label='Light Smoke', alpha=0.75)
    ]

ax.legend(handles=legend_elements, loc='upper right')

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Voyager)
ax.set_axis_off()
ax.set_title(pd.to_datetime(jan8_date).strftime('%Y-%m-%d'), fontsize=16)

total_bounds = zctas_kp.total_bounds
ax.set_xlim(total_bounds[0], total_bounds[2])
ax.set_ylim(total_bounds[1], total_bounds[3])

plt.tight_layout()
plt.show()